# Training Analysis

This notebook inspects the training process, dataset statistics, model size, loss curves, and validation metrics.

In [ ]:
from pathlib import Path
import json
import torch
from datasets import load_dataset
from config import get_config
from tokenizer import get_or_build_tokenizer
from visualization import plot_training_history, plot_validation_metrics, plot_translation_lengths

In [ ]:
config = get_config()
dataset = load_dataset(config['datasource'], f"{config['lang_src']}-{config['lang_tgt']}", split='train')
tokenizer_src = get_or_build_tokenizer(config, dataset, config['lang_src'])
tokenizer_tgt = get_or_build_tokenizer(config, dataset, config['lang_tgt'])
len(dataset), tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()

In [ ]:
source_lengths = []
target_lengths = []
for item in dataset:
    source_lengths.append(len(tokenizer_src.encode(item['translation'][config['lang_src']]).ids))
    target_lengths.append(len(tokenizer_tgt.encode(item['translation'][config['lang_tgt']]).ids))
plot_translation_lengths(source_lengths, target_lengths)

In [ ]:
history_path = Path(config.get('history_file', 'runs/history.json'))
history = json.loads(history_path.read_text()) if history_path.exists() else {'train_loss': [], 'val_loss': [], 'cer': [], 'wer': [], 'bleu': []}
if history['train_loss']:
    plot_training_history(history)
if any(history.get(key) for key in ('cer', 'wer', 'bleu')):
    plot_validation_metrics(history)

In [ ]:
from model import build_transformer
model = build_transformer(
    tokenizer_src.get_vocab_size(),
    tokenizer_tgt.get_vocab_size(),
    config['seq_len'],
    config['seq_len'],
    d_model=config['d_model'],
    N=config['num_layers'],
    h=config['num_heads'],
    dropout=config['dropout'],
    d_ff=config['d_ff']
)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
parameter_count, trainable_count